In [ ]:
#
# Project:
#      PyTorch Dojo (https://github.com/wo3kie/ml-dojo)
#
# Author:
#      Lukasz Czerwinski (https://www.lukaszczerwinski.pl/)
#

$$ \text{Definition} \; \sigma $$

$$ \mathbb{R} \to \mathbb{R}, \quad p(z) = \frac{e^z}{1+e^z} $$

$$ 
\mathbb{R^n} \to \mathbb{R^n}, \quad p(z) =
(p(z_1), p(z_2), \dots, p(z_n)) 
$$

$$ \text{Derivative} $$

$$ \frac{dp}{dz_i} = p_i(1 - p_i) $$

$$ \frac{dp}{dz} = p  \odot (1 - p) $$

$$ \text{Jacobian} $$

$$
J_p(z) =
\begin{bmatrix}
    \frac{dp}{dz_1} & 0 & \cdots & 0 \\
    0 & \frac{dp}{dz_2} & \cdots & 0 \\
    \vdots & \vdots & \ddots & \vdots \\
    0 & 0 & \cdots & \frac{dp}{dz_n}
\end{bmatrix}
$$

$$ dp = J_p(z) \, dz $$

$$ \text{Frobenius equality} $$

$$
\left \langle \frac{dF}{dz}, dz \right \rangle =
\left \langle \frac{dF}{dy}, dp \right \rangle \\[2em]

\left \langle \frac{dF}{dp}, dp \right \rangle = \\[0.5em]
\left \langle \frac{dF}{dp}, J_y(z) \, dz \right \rangle = \\[0.5em]
\left \langle {J_y(z)}^\top \, \frac{dF}{dp}, dz \right \rangle \implies \\[1em]

\frac{dF}{dz} = {J_y(z)}^\top \, \frac{dF}{dp} \\[0.5em]
\frac{dF}{dz} = \frac{dp}{dz} \odot \frac{dF}{dp}
$$

In [ ]:
import torch as tr
import torch.nn as nn
import torch.autograd as autograd

import import_ipynb
from approx import approx # type: ignore


def _sigmoid_neg(z: tr.Tensor) -> tr.Tensor:
    """Numerically stable `sigmoid` for negative inputs."""

    exp = tr.exp(z)
    return exp / (exp + 1)


def _sigmoid_pos(z: tr.Tensor) -> tr.Tensor:
    """Numerically stable `sigmoid` for positive inputs."""

    return 1 / (1 + tr.exp(-z))


def sigmoid(z: tr.Tensor) -> tr.Tensor:
    """Numerically stable `sigmoid` function."""

    z = tr.as_tensor(z, dtype=tr.float32)

    mask = (z >= 0)
    p = tr.empty_like(z)
    p[mask] = _sigmoid_pos(z[mask])
    p[~mask] = _sigmoid_neg(z[~mask])

    return p


class SigmoidFunction(autograd.Function):
    """Custom autograd function for the `sigmoid`."""

    @staticmethod
    def forward(ctx, z: tr.Tensor) -> tr.Tensor:
        p = sigmoid(z)
        ctx.save_for_backward(p)
        return p

    @staticmethod
    def backward(ctx, dF_dp: tr.Tensor) -> tuple[tr.Tensor, ]:
        (p, ) = ctx.saved_tensors

        dp_dz = p * (1 - p)
        dF_dz = dp_dz * dF_dp

        return (dF_dz, )


class Sigmoid(nn.Module):
    """Custom module for the `sigmoid`."""

    def __init__(self) -> None:
        super().__init__()

    def forward(self, z: tr.Tensor) -> tr.Tensor:
        return SigmoidFunction.apply(z)

In [ ]:
# Unit tests

def test_basic() -> None:
    assert sigmoid(1000) == approx(1.0)
    assert sigmoid(-1000) == approx(0.0)
    assert sigmoid(0) == approx(0.5)
    assert sigmoid(1) == approx(0.731, atol=0.001, rtol=0)
    assert sigmoid(-1) == approx(0.269, atol=0.001, rtol=0)


def test_gradcheck(samples) -> None:
    def func(z: tr.Tensor) -> tr.Tensor:
        return SigmoidFunction.apply(z)

    z = tr.randn(samples, dtype=tr.float64, requires_grad=True)
    assert autograd.gradcheck(func, (z, ), eps=0.001, atol=0.001)


def test_compare(samples) -> None:
    z = tr.randn(samples, dtype=tr.float32, requires_grad=True)

    z1 = z.clone().detach().requires_grad_(True)
    p1 = Sigmoid()(z1)
    F1 = p1.sum()
    F1.backward()

    z2 = z.clone().detach().requires_grad_(True)
    p2 = tr.sigmoid(z2)
    F2 = p2.sum()
    F2.backward()

    assert p1 == approx(p2)
    assert z1.grad == approx(z2.grad)


if __name__ == "__main__":
    test_basic()

    test_gradcheck(1)
    test_gradcheck(10)
    test_gradcheck(100)

    test_compare(1)
    test_compare(10)
    test_compare(100)